### eNextSUT generator

Run this code to generate (or update) our eNextSUT reference database.
By default, it relies on Exiobase Hybrid version (v3.3.18) but could be potentially arranged to any other SUT in the future. 

As if 5th November 2024, the above-mentioned database is parsed, aggregated in terms of electricity commodity (1 commodity) and activities (EMBER power plants technologies) and electricity production mixes are updated according to EMBER ones for a given year.

Just mind to update your paths in the 'paths.yml' file and to specify the user (your initials) and the desired electricity mix in the first cell, then run all the code

In [ ]:
import mario
import yaml
import pandas as pd
from support.ember_remapping import map_ember_to_classification

user = 'LR'   # change this to your username
year = 2023   # change this to the year you want to update the electricity mixes to

with open('paths.yml', 'r') as file: # open the yml file
    paths = yaml.safe_load(file)

paths = paths[user]

In [ ]:
# Parse raw SUT
world = mario.parse_from_txt(paths['raw'], table='SUT', mode='flows')

In [ ]:
# Aggregate electricity commodities and activities to match EMBER
world.aggregate("support/aggregate_ee.xlsx",ignore_nan=True)

In [ ]:
# Parse ember electricity generation data, map to exiobase and get electricity mix for a given year 
ee_mix = map_ember_to_classification(
    path = paths['ember'],
    classification = 'EXIO3',
    year = year,
    mode = 'mix',
)

In [ ]:
#%% Update electricity mixes
z = world.z
s = world.s

for region in world.get_index('Region'):
    print(region,end=' ')
    new_mix = ee_mix.loc[(region,slice(None),slice(None)),'Value'].to_frame().sort_index(axis=0) 
    new_mix.index = new_mix.index.get_level_values(2)
    old_market_share = s.loc[(region, 'Activity', new_mix.index),(region,'Commodity','Electricity')].sum().sum()
    
    s.loc[(region, 'Activity', new_mix.index),(region,'Commodity','Electricity')] = new_mix.values*old_market_share # check if commodity electricity is called "Electricity" in aggregation excel file
    # s.loc[:,(region,'Commodity','Electricity')] /= s.loc[:,(region,'Commodity','Electricity')].sum()
    print('done')

z.update(s)

world.update_scenarios('baseline',z=z)
world.reset_to_coefficients('baseline')

In [ ]:
world.to_txt(paths['export'])

In [ ]:
ghgs = {
    'Carbon dioxide, fossil (air - Emiss)':1,
    'CH4 (air - Emiss)':29,
    'N2O (air - Emiss)':273
    }

f = world.f.loc[ghgs.keys(),(slice(None),'Commodity','Electricity')]
if isinstance(f,pd.Series):
    f = f.to_frame()

for ghg,gwp in ghgs.items():
    f.loc[ghg,:] *= gwp

f = f.sum(0)*3.6
# f = f.to_frame()   
f 
# f.reset_index(inplace=True)
# f.columns = ['Region','Item','Activity','Value']
# f = f.drop('Item',axis=1)
# f.set_index(['Region','Activity'],inplace=True)
# f = f.unstack()
# f = f.droplevel(0,axis=1)
# f.to_clipboard()


In [ ]:
ee_prod = world.s.loc[:,('IT','Commodity','Electricity')]
ee_prod = ee_prod.to_frame()
ee_prod.columns = ['EE_market_share']
ee_prod = ee_prod.query('EE_market_share>0')
ee_prod.sort_values('EE_market_share',ascending=False,inplace=True)
ee_prod


ghgs = {
    'Carbon dioxide, fossil (air - Emiss)':1,
    'CH4 (air - Emiss)':29,
    'N2O (air - Emiss)':273
    }

f = world.f.loc[ghgs.keys(),('IT','Activity',ee_prod.index.get_level_values(2))]
if isinstance(f,pd.Series):
    f = f.to_frame()

for ghg,gwp in ghgs.items():
    f.loc[ghg,:] *= gwp

f = f.sum(0)*3.6
f = f.droplevel(0)
f = f.droplevel(0)
f = f.to_frame()
f.columns = ['gCO2eq/kWh']

ee_prod = ee_prod.join(f)
ee_prod.to_clipboard()